# 05 — Pull the full Landsat time series from Earth Engine

**Purpose.** Downloads every usable Landsat 5, 7 and 8 scene from 1980 to 2024 for the glaciers chosen
for inference: the 250-per-region selection, 1,024 glaciers. This is the imagery the model runs on to
measure area change. `EePull` applies `CLOUD_COVER <= 10` and image quality 9 over each bounding box,
writes one 30 m GeoTIFF per scene, and logs a line per scene; watch the log grow to confirm the pull is
still running. A satellite that fails for a glacier is logged as a warning and skipped.

The pull resumes: glaciers that already have a scene folder are skipped. To pull onto an external
disk, start Jupyter with `GV_DATA_ROOT` pointing at it.

## Inputs
- **`config.inference_sample_path(250)`**: the glacier list, the saved Athena output of
  `sql/identify_inference_glims_ids_geog_area_rollup_250.sql`. Gitignored.
- **`config.glims_selection_path()`**: bounding boxes from notebook 01.
- **Earth Engine account**, as in notebook 03. A long, network-bound run.

## Outputs
Under `config.landing_zone(config.INFERENCE_DATA_LABEL)` (gitignored):
- `landsat/<glims_id>/<glims_id>_<date>_<L5|L7|L8>_C02_T1_L2_SR.tif`: one file per scene.
- `landsat/<glims_id>/meta_data/metadata_list_l{5,7,8}`: raw Earth Engine metadata, read by notebook 07.
- `logs/log_geog_area_rollup_250.log`: one line per scene, read by notebook 07.

## Manual step afterwards (not in code)
The canonical copy lives in `s3://full-time-series-images-t1-l2-sr` (us-west-1), under `<glims_id>/`.
Upload new pulls there. See `.agents/references/aws.md`.

In [ ]:
from datetime import datetime

import ee
import pandas as pd
from tqdm import tqdm

from glacierview import config, glims, log, metadata
from glacierview.earthengine import EePull

log.configure()
ee.Authenticate()
ee.Initialize(project="psyched-upgrade-377000")

In [ ]:
LABEL = config.INFERENCE_DATA_LABEL
START, END = datetime(1980, 1, 1), datetime(2024, 12, 31)

pull = EePull(config.log_dir(LABEL), "geog_area_rollup_250")
landsat = config.landsat_dir(LABEL)
done = set(metadata.glacier_ids(landsat)) if landsat.exists() else set()

chosen = pd.read_csv(config.inference_sample_path(250))["glims_id"]
glaciers = glims.read_selection()
glaciers = glaciers[glaciers["glac_id"].isin(chosen) & ~glaciers["glac_id"].isin(done)]
print(f"{len(glaciers)} glaciers to pull, {len(done)} already done")

In [ ]:
for glacier in tqdm(glaciers.itertuples(), total=len(glaciers)):
    pull.export_all_landsat(
        glacier.glac_id,
        glims.parse_bounding_box(glacier.bboxes),
        START,
        END,
        config.glacier_dir(glacier.glac_id, LABEL),
    )